# DuckDB + Parquet: Python Tutorial

This notebook introduces working with research data using DuckDB and Parquet in Python.
It uses a small synthetic dataset — no real patient data.

**What we cover:**
1. Environment check
2. First-run setup: CSV → Parquet
3. Querying Parquet directly with DuckDB (no database needed)
4. Creating a persistent DuckDB database
5. Basic queries — SQL and pandas
6. Joining tables
7. Researcher pattern: creating a filtered working subset

**Prerequisites:** run `uv sync && uv run jupyter lab` from the repo root.

## 1. Environment check

In [ ]:
import sys
import duckdb
import pandas as pd
import pyarrow

print(f"Python:  {sys.version.split()[0]}")
print(f"DuckDB:  {duckdb.__version__}")
print(f"pandas:  {pd.__version__}")
print(f"pyarrow: {pyarrow.__version__}")

## 2. First-run setup: CSV → Parquet

Run this cell once. It reads the source CSVs from `data/raw_csv/` and writes
Parquet files to `data/parquet/`. Subsequent cells read from Parquet directly.

This mirrors the real-world pattern where IT migrates legacy data to Parquet
once, and researchers query from that central store.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

RAW_DIR     = Path("../../data/raw_csv")
PARQUET_DIR = Path("../../data/parquet")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

for csv_file in sorted(RAW_DIR.glob("*.csv")):
    df       = pd.read_csv(csv_file, parse_dates=True)
    out_path = PARQUET_DIR / csv_file.with_suffix(".parquet").name
    pq.write_table(
        pa.Table.from_pandas(df, preserve_index=False),
        out_path,
        compression="snappy"
    )
    print(f"{csv_file.name:30s} → {out_path.name}  ({len(df):,} rows)")

## 3. Query Parquet directly — no database needed

DuckDB can query Parquet files in place. This is useful for quick exploration
before deciding whether to load data into a database.

In [ ]:
# In-memory connection — nothing written to disk
con = duckdb.connect()

# Inspect a Parquet file without loading it first
con.execute(f"""
    SELECT *
    FROM read_parquet('{PARQUET_DIR}/patients.parquet')
    LIMIT 5
""").df()

In [ ]:
# Aggregate directly on Parquet — no import step
con.execute(f"""
    SELECT
        diagnosis_year,
        sex,
        COUNT(*) AS n_patients
    FROM read_parquet('{PARQUET_DIR}/patients.parquet')
    GROUP BY diagnosis_year, sex
    ORDER BY diagnosis_year, sex
""").df()

In [ ]:
con.close()

## 4. Create a persistent DuckDB database

For repeated work, load data into a `.duckdb` file. Queries are faster,
and the file is self-contained — easy to share or archive with a project.

In [ ]:
DB_PATH = Path("../../data/tutorial.duckdb")

con = duckdb.connect(str(DB_PATH))

# Load each Parquet file as a table
for pq_file in sorted(PARQUET_DIR.glob("*.parquet")):
    table_name = pq_file.stem   # e.g. patients, visits, medications
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_parquet('{pq_file}')
    """)
    n = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"  {table_name:20s} {n:,} rows")

print("\nTables in database:")
con.execute("SHOW TABLES").df()

## 5. Basic queries

In [ ]:
# How many patients per diagnosis code?
con.execute("""
    SELECT
        diagnosis_code,
        COUNT(*)           AS n_patients,
        MIN(diagnosis_year) AS first_seen,
        MAX(diagnosis_year) AS last_seen
    FROM patients
    GROUP BY diagnosis_code
    ORDER BY n_patients DESC
""").df()

In [ ]:
# Filter: Icelandic patients diagnosed from 2015 onwards
con.execute("""
    SELECT *
    FROM patients
    WHERE country_code   = 'IS'
      AND diagnosis_year >= 2015
    ORDER BY diagnosis_year, patient_id
""").df()

In [ ]:
# Visit counts and types
con.execute("""
    SELECT
        visit_type,
        clinic,
        COUNT(*) AS n_visits
    FROM visits
    GROUP BY visit_type, clinic
    ORDER BY n_visits DESC
""").df()

## 6. Joining tables

DuckDB supports full SQL — joins work exactly as you'd expect.

In [ ]:
# Patients with their visit counts and distinct drugs
con.execute("""
    SELECT
        p.patient_id,
        p.sex,
        p.diagnosis_code,
        p.diagnosis_year,
        COUNT(DISTINCT v.visit_id)    AS n_visits,
        COUNT(DISTINCT m.drug_name)   AS n_distinct_drugs
    FROM patients p
    LEFT JOIN visits      v ON p.patient_id = v.patient_id
    LEFT JOIN medications m ON p.patient_id = m.patient_id
    GROUP BY p.patient_id, p.sex, p.diagnosis_code, p.diagnosis_year
    ORDER BY n_visits DESC
    LIMIT 10
""").df()

In [ ]:
# Most commonly prescribed drugs per diagnosis
con.execute("""
    SELECT
        p.diagnosis_code,
        m.drug_name,
        COUNT(*) AS n_prescriptions
    FROM medications m
    JOIN patients p USING (patient_id)
    GROUP BY p.diagnosis_code, m.drug_name
    ORDER BY p.diagnosis_code, n_prescriptions DESC
""").df()

## 7. Researcher pattern: filtered working subset

In production, researchers don't get direct access to the central Parquet files.
Instead, IT provides a script to create a filtered working `.duckdb` containing
only the data relevant to their project.

Here's what that looks like from the researcher's side:

In [ ]:
WORKING_DB = Path("../../data/my_project.duckdb")

work_con = duckdb.connect(str(WORKING_DB))

# Pull only what this project needs — Icelandic patients, 2015+
work_con.execute(f"""
    CREATE OR REPLACE TABLE patients AS
    SELECT *
    FROM read_parquet('{PARQUET_DIR}/patients.parquet')
    WHERE country_code   = 'IS'
      AND diagnosis_year >= 2015
""")

# Bring in only visits for those patients
work_con.execute(f"""
    CREATE OR REPLACE TABLE visits AS
    SELECT v.*
    FROM read_parquet('{PARQUET_DIR}/visits.parquet') v
    WHERE v.patient_id IN (SELECT patient_id FROM patients)
""")

n_p = work_con.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
n_v = work_con.execute("SELECT COUNT(*) FROM visits").fetchone()[0]
print(f"Working subset: {n_p} patients, {n_v} visits")
print(f"Saved to: {WORKING_DB}")

work_con.close()

In [ ]:
# Always close the main connection when done
con.close()

## Next steps

- See the R notebook (`notebooks/r/01_intro_duckdb_parquet.ipynb`) for the same
  workflow in R using DBI and dplyr
- The migration toolkit (separate repo) shows how legacy Excel/Access files get
  converted to the Parquet files used here